# 🏔️ PIPELINE ICEBERG V4 — SCORING TERRITORIAL BANCAIRE
### Production-Ready · Source médicale en amont · Départements 91 & 94

**Sources (ordre d'exécution)** :
1. 🏥 **BPE INSEE** (médecins, pharmacies, hôpitaux) — **source principale locale**
2. 🌐 Geo API — communes, population, GPS
3. 🏢 SIRENE INSEE — établissements actifs/fermés
4. 🚆 IDF Mobilités / SNCF / OSM — transport
5. 🏠 DVF — marché immobilier

**Scores produits (8)** : fragilité · émergence · freins invisibles · momentum · potentiel investissement · risque crédit · désert médical · désert commercial

---
**Installation** : `pip install pandas numpy requests openpyxl`

In [ ]:
!pip install pandas numpy requests openpyxl -q

## ⚙️ PARTIE 0 : Imports, Configuration & Logging

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════╗
║   PIPELINE ICEBERG V4 — SCORING TERRITORIAL BANCAIRE                   ║
║   Architecture repensée : source médicale intégrée EN AMONT            ║
╠══════════════════════════════════════════════════════════════════════════╣
║   ORDRE D'EXÉCUTION :                                                  ║
║     0. Config & Logging                                                 ║
║     1. Chargement BPE médecins (fichier local ou API INSEE)             ║
║     2. Collecte API : Geo communes, SIRENE, Transport, DVF              ║
║     3. Fusion & nettoyage (BPE comme ancre)                             ║
║     4. Feature engineering                                              ║
║     5. Scoring bancaire (8 scores dont 2 déserts)                      ║
║     6. Segmentation territoriale                                        ║
║     7. Export JSON / CSV / Excel + rapport                              ║
╠══════════════════════════════════════════════════════════════════════════╣
║   Sources réelles : BPE INSEE · Geo API · SIRENE · IDF Mobilités · DVF ║
║   Fallback simulé : données réalistes si source indisponible            ║
╚══════════════════════════════════════════════════════════════════════════╝
"""

from __future__ import annotations

import json
import logging
import sys
import time
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import requests

# ============================================================================
# CONFIGURATION PRINCIPALE
# ============================================================================

CONFIG: Dict = {
    # Départements à analyser
    "departements": ["91", "94"],

    # --- SOURCE PRINCIPALE : fichier BPE médecins (local ou URL) ---
    # Priorité 1 : chemin local vers votre fichier CSV/JSON médecins
    # Laisser vide pour utiliser l'API BPE INSEE ou le fallback simulé
    "bpe_fichier_local": "",           # Ex: "data/medecins_91_94.csv"
    "bpe_separateur":    ";",          # Séparateur CSV (souvent ';' pour INSEE)

    # Clé API SIRENE (inscription gratuite : https://portail-api.insee.fr/)
    "sirene_api_key": "d4f6bab8-83a3-4a77-b6ba-b883a33a770c",

    # Chemins de sortie
    "output_json":   "iceberg_dataset_v4.json",
    "output_csv":    "iceberg_dataset_v4.csv",
    "output_excel":  "iceberg_dataset_v4.xlsx",
    "output_report": "iceberg_rapport_v4.txt",

    # Paramètres réseau
    "api_timeout":        15,
    "sirene_max_results": 2000,

    # Si True : génère des données réalistes pour les sources indisponibles
    "use_fallback_data": True,
}

# ============================================================================
# LOGGING
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("pipeline_iceberg_v4.log", encoding="utf-8"),
    ],
)
log = logging.getLogger("iceberg_v4")

## 🌐 PARTIE 1 : Helpers réseau

In [ ]:
def get_with_retry(
    url: str,
    headers: dict = None,
    params:  dict = None,
    timeout: int  = 15,
    retries: int  = 3,
) -> Optional[dict | list]:
    """
    Appel HTTP GET avec retry exponentiel (2s, 4s, 8s).
    Retourne None si tous les essais échouent.
    Ne raise jamais — le pipeline continue toujours.
    """
    headers = headers or {}
    params  = params  or {}

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.Timeout:
            log.warning(f"Timeout ({attempt}/{retries}) → {url}")
        except requests.exceptions.HTTPError as e:
            code = e.response.status_code
            log.warning(f"HTTP {code} ({attempt}/{retries}) → {url}")
            if code in (400, 401, 403, 404):
                break  # Erreur client → inutile de réessayer
        except requests.exceptions.ConnectionError:
            log.warning(f"Connexion échouée ({attempt}/{retries}) → {url}")
        except Exception as e:
            log.warning(f"Erreur inattendue ({attempt}/{retries}) : {e}")

        if attempt < retries:
            wait = 2 ** attempt
            log.info(f"  Nouvelle tentative dans {wait}s…")
            time.sleep(wait)

    log.error(f"Échec définitif : {url}")
    return None


def get_csv_with_retry(
    url: str,
    sep: str = ";",
    timeout: int = 15,
    usecols: list = None,
) -> pd.DataFrame:
    """
    Télécharge et parse un CSV distant.
    Retourne DataFrame vide si échec — ne bloque jamais le pipeline.
    usecols : list optionnel pour limiter les colonnes chargées (optimisation mémoire).
    """
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        read_kwargs = {"sep": sep, "low_memory": False}
        if usecols:
            # Lecture header d'abord pour mapper les colonnes disponibles
            header_df = pd.read_csv(StringIO(r.text), sep=sep, nrows=0)
            available = [c for c in usecols if c in header_df.columns]
            if available:
                read_kwargs["usecols"] = available
        df = pd.read_csv(StringIO(r.text), **read_kwargs)
        df.columns = [c.lower().strip().replace(" ", "_").replace("-", "_") for c in df.columns]
        return df
    except Exception as e:
        log.warning(f"CSV distant indisponible ({url[:70]}…) : {e}")
        return pd.DataFrame()

## 🏥 PARTIE 2 : SOURCE PRINCIPALE — Chargement & traitement des données médecins (BPE)

> **Pourquoi en premier ?**  
> Le fichier BPE est la source la plus structurante du pipeline v4 : il définit les déserts médicaux qui pondèrent directement `score_fragilite` et `score_freins_invisibles`. Il est chargé **avant** tout appel API pour éviter des requêtes inutiles si la source locale échoue, et pour servir de référence de validation des communes.

In [ ]:
# ── URL officielle du dataset BPE Santé IDF ──────────────────────────────────
# Source : https://data.iledefrance.fr/explore/dataset/
#          les-service-de-sante-par-commune-ou-par-arrondissement-base-permanente-des-equip/
BPE_IDF_URL = (
    "https://data.iledefrance.fr/api/explore/v2.1/catalog/datasets/"
    "les-service-de-sante-par-commune-ou-par-arrondissement-base-permanente-des-equip/"
    "exports/csv?lang=fr&timezone=Europe%2FParis&delimiter=%3B"
)

# Correspondance colonnes du dataset IDF → noms internes du pipeline
# (noms exacts tels qu'exportés par data.iledefrance.fr)
BPE_IDF_MAPPING = {
    # Établissements
    "nb_equip_d201":  "nb_pharmacie",
    "nb_equip_d221":  "nb_hopital",            # Établissement santé court séjour
    "nb_equip_d222":  "nb_hopital_moyen",       # Moyen séjour
    "nb_equip_d231":  "nb_urgences",
    "nb_equip_d302":  "nb_centre_sante",
    "nb_equip_d303":  "nb_maternite",
    # Colonnes possibles selon version du dataset
    "pharmacie":                  "nb_pharmacie",
    "etablissement_sante_court":  "nb_hopital",
    "urgences":                   "nb_urgences",
    "maternite":                  "nb_maternite",
    "centre_sante":               "nb_centre_sante",
    "laboratoire":                "nb_laboratoire",
    "ambulance":                  "nb_ambulance",
}

# Colonnes BPE attendues dans le pipeline après chargement
BPE_COLONNES_MEDICALES = [
    "nb_medecin_generaliste", "nb_medecin_specialiste",
    "nb_pharmacie", "nb_hopital", "nb_urgences",
    "nb_infirmier", "nb_dentiste", "nb_kinesitherapeute",
    "medecins_10k_hab", "est_desert_medical",
]

BPE_COLONNES_COMMERCIALES = [
    "nb_supermarche", "nb_epicerie", "nb_boulangerie",
    "nb_banque", "nb_poste", "nb_boucherie",
    "nb_ecole_primaire", "nb_college", "nb_lycee",
    "est_desert_commercial",
]

BPE_COLONNES_TOUTES = BPE_COLONNES_MEDICALES + BPE_COLONNES_COMMERCIALES


def _detect_bpe_columns(df: pd.DataFrame) -> Dict[str, str]:
    """
    Détecte automatiquement les colonnes BPE présentes dans un DataFrame,
    quelle que soit la version ou l'origine du fichier (IDF, INSEE, export custom).
    Retourne un dict {colonne_source → colonne_pipeline}.
    """
    cols = {c.lower().strip() for c in df.columns}
    mapping = {}

    # Colonnes déjà au format pipeline (ex: export précédent du pipeline)
    for col in BPE_COLONNES_TOUTES:
        if col in cols:
            mapping[col] = col

    # Colonnes format IDF (nb_equip_DXXX ou noms littéraux)
    for src, dst in BPE_IDF_MAPPING.items():
        if src in cols and dst not in mapping.values():
            mapping[src] = dst

    return mapping


def _load_csv_auto(filepath_or_url: str, sep: str = None) -> pd.DataFrame:
    """
    Charge un CSV depuis un chemin local ou une URL distante.
    Détecte automatiquement le séparateur si non précisé (virgule ou point-virgule).
    """
    is_url = filepath_or_url.startswith("http")
    try:
        if is_url:
            log.info(f"  [BPE] Téléchargement : {filepath_or_url[:80]}…")
            r = requests.get(filepath_or_url, timeout=30)
            r.raise_for_status()
            content = r.text
        else:
            log.info(f"  [BPE] Lecture locale : {filepath_or_url}")
            with open(filepath_or_url, encoding="utf-8", errors="replace") as f:
                content = f.read()

        # Détection auto du séparateur
        if sep is None:
            first_line = content.split("\n")[0]
            sep = ";" if first_line.count(";") > first_line.count(",") else ","

        df = pd.read_csv(
            StringIO(content), sep=sep, low_memory=False,
            dtype=str,  # tout en str pour éviter les cast automatiques incorrects
        )
        df.columns = [c.lower().strip().replace(" ", "_").replace("-", "_") for c in df.columns]
        log.info(f"  [BPE] Chargé : {len(df):,} lignes × {len(df.columns)} colonnes (sep='{sep}')")
        return df

    except Exception as e:
        log.error(f"  [BPE] Erreur chargement ({filepath_or_url[:60]}) : {e}")
        return pd.DataFrame()


def _extract_bpe_columns(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Extrait et normalise les colonnes BPE depuis un DataFrame brut.
    Fonctionne quel que soit le format source :
      - export CSV du pipeline (colonnes déjà nommées nb_medecin_*)
      - export IDF (colonnes nb_equip_DXXX ou noms littéraux)
      - BPE INSEE brut (CODGEO + TYPEQU + NB_EQUIP)

    Retourne un DataFrame [code_commune] + colonnes BPE normalisées.
    """
    if df_raw.empty:
        return pd.DataFrame()

    # Clé de jointure commune
    col_commune = next(
        (c for c in ["code_commune", "codgeo", "depcom", "com_arm"] if c in df_raw.columns), None
    )
    if col_commune is None:
        log.error("  [BPE] Colonne code commune introuvable (cherché: code_commune, codgeo, depcom)")
        return pd.DataFrame()

    df_raw = df_raw.rename(columns={col_commune: "code_commune"})
    df_raw["code_commune"] = df_raw["code_commune"].astype(str).str.strip().str.zfill(5)

    # ── Cas 1 : format BPE brut INSEE (une ligne par équipement : TYPEQU + NB_EQUIP) ──
    col_type = next((c for c in ["typequ", "type_equip", "typeequip"] if c in df_raw.columns), None)
    if col_type:
        log.info("  [BPE] Format détecté : BPE brut INSEE (TYPEQU + NB_EQUIP)")
        TYPES_CIBLES = {
            "D101": "nb_medecin_generaliste", "D102": "nb_medecin_specialiste",
            "D103": "nb_infirmier",           "D104": "nb_dentiste",
            "D105": "nb_kinesitherapeute",    "D106": "nb_sage_femme",
            "D201": "nb_pharmacie",           "D221": "nb_hopital",
            "D231": "nb_urgences",            "B201": "nb_supermarche",
            "B202": "nb_epicerie",            "B203": "nb_boulangerie",
            "B204": "nb_boucherie",           "A504": "nb_banque",
            "A601": "nb_poste",               "C101": "nb_ecole_primaire",
            "C201": "nb_college",             "C301": "nb_lycee",
        }
        col_nb = next((c for c in ["nb_equip", "nb", "nombre", "effectif"] if c in df_raw.columns), None)
        df_raw["_nb"] = pd.to_numeric(df_raw[col_nb], errors="coerce").fillna(1) if col_nb else 1
        df_raw["_typequ_up"] = df_raw[col_type].astype(str).str.upper().str.strip()
        df_raw["_nom_equip"] = df_raw["_typequ_up"].map(TYPES_CIBLES)
        df_filt = df_raw[df_raw["_nom_equip"].notna()].copy()

        if df_filt.empty:
            log.warning("  [BPE] Aucun type d'équipement reconnu dans TYPEQU")
            return pd.DataFrame()

        pivot = (
            df_filt.groupby(["code_commune", "_nom_equip"])["_nb"]
            .sum().unstack(fill_value=0).reset_index()
        )
        pivot.columns.name = None
        log.info(f"  [BPE] Pivot : {len(pivot)} communes")
        pivot["donnees_bpe_simulees"] = False
        return pivot

    # ── Cas 2 : format tabulaire (une ligne par commune, colonnes = types d'équipements) ──
    log.info("  [BPE] Format détecté : tabulaire (une ligne par commune)")
    col_mapping = _detect_bpe_columns(df_raw)

    if not col_mapping:
        log.warning("  [BPE] Aucune colonne BPE reconnue dans le fichier")
        return pd.DataFrame()

    cols_a_extraire = list(col_mapping.keys())
    df_bpe = df_raw[["code_commune"] + cols_a_extraire].copy()
    df_bpe = df_bpe.rename(columns=col_mapping)

    # Déduplication si plusieurs sources mappent vers la même colonne
    df_bpe = df_bpe.loc[:, ~df_bpe.columns.duplicated()]

    # Conversion numérique
    for col in df_bpe.columns:
        if col != "code_commune":
            df_bpe[col] = pd.to_numeric(df_bpe[col], errors="coerce").fillna(0)

    df_bpe["donnees_bpe_simulees"] = False
    log.info(f"  [BPE] Colonnes extraites ({len(col_mapping)}) : {list(col_mapping.values())}")
    return df_bpe


def load_and_prepare_bpe(
    departements: List[str],
    bpe_fichier_local: str = "",
    bpe_separateur: str = None,
    timeout: int = 15,
) -> pd.DataFrame:
    """
    POINT D'ENTRÉE PRINCIPAL — données médecins/BPE.
    Exécuté EN PREMIER dans le pipeline, avant tout appel API externe.

    Stratégie (ordre de priorité) :
      1. Fichier CSV local (bpe_fichier_local) — zéro appel réseau
         Supporte : colonnes nb_medecin_* déjà nommées OU format brut INSEE
      2. API open data IDF (data.iledefrance.fr) — dataset BPE santé officiel
      3. DataFrame vide → fallback simulé déclenché dans merge_all_sources()

    Args:
        departements       : liste des codes dept (ex: ["91", "94"])
        bpe_fichier_local  : chemin local vers CSV médecins (optionnel)
        bpe_separateur     : séparateur CSV — None = détection automatique
        timeout            : délai réseau en secondes

    Returns:
        DataFrame [code_commune, nb_medecin_generaliste, nb_pharmacie, …]
        prêt à être jointé dans merge_all_sources().
    """
    log.info("[ÉTAPE 1/7] Chargement BPE — données médicales & équipements")

    # ── Priorité 1 : fichier local ────────────────────────────────────────────
    if bpe_fichier_local:
        df_raw = _load_csv_auto(bpe_fichier_local, sep=bpe_separateur)
        if not df_raw.empty:
            df_bpe = _extract_bpe_columns(df_raw)
            if not df_bpe.empty:
                # Filtrage sur les départements cibles si la colonne est présente
                if "code_dept" in df_raw.columns or any(
                    df_bpe["code_commune"].str[:2].isin(departements)
                ):
                    df_bpe = df_bpe[df_bpe["code_commune"].str[:2].isin(departements)]
                log.info(f"  ✅ BPE depuis fichier local : {len(df_bpe)} communes")
                return df_bpe

    # ── Priorité 2 : API open data IDF ───────────────────────────────────────
    log.info("  Fichier local absent/invalide → API data.iledefrance.fr…")
    df_raw = _load_csv_auto(BPE_IDF_URL, sep=";")
    if not df_raw.empty:
        df_bpe = _extract_bpe_columns(df_raw)
        if not df_bpe.empty:
            # Filtrage sur les départements 91 et 94
            df_bpe = df_bpe[df_bpe["code_commune"].str[:2].isin(departements)]
            log.info(f"  ✅ BPE depuis API IDF : {len(df_bpe)} communes")
            return df_bpe

    # ── Priorité 3 : fallback signalé ────────────────────────────────────────
    log.warning("  ⚠️  BPE indisponible — données simulées dans merge_all_sources()")
    return pd.DataFrame()


## 🌍 PARTIE 3 : Collecte des données externes (API)

In [ ]:
def collect_communes(departements: List[str], timeout: int = 15) -> pd.DataFrame:
    """
    Collecte les communes via l'API Geo gouvernementale.
    Source : https://geo.api.gouv.fr/
    Données : nom, code INSEE, population, surface (km²), GPS
    Coût    : gratuit, sans clé API
    """
    log.info(f"[COLLECTE] Communes — {len(departements)} département(s)")
    frames = []

    for dept in departements:
        data = get_with_retry(
            f"https://geo.api.gouv.fr/departements/{dept}/communes",
            params={
                "fields":   "nom,code,population,surface,centre,codesPostaux,departement",
                "format":   "json",
                "geometry": "centre",
            },
            timeout=timeout,
        )
        if not data:
            log.warning(f"  Dept {dept} : API Geo indisponible")
            continue

        rows = []
        for c in data:
            coords = c.get("centre", {}).get("coordinates", [None, None])
            rows.append({
                "code_commune":    str(c.get("code", "")).strip(),
                "ville":           str(c.get("nom",  "")).strip(),
                "code_dept":       dept,
                "nom_departement": c.get("departement", {}).get("nom", ""),
                "population":      int(c["population"]) if c.get("population") else 0,
                "surface_km2":     float(c["surface"])  if c.get("surface")    else 0.0,
                "code_postal":     (c.get("codesPostaux") or [None])[0],
                "latitude":        coords[1] if coords and len(coords) > 1 else None,
                "longitude":       coords[0] if coords and len(coords) > 0 else None,
            })

        if rows:
            df_dept = pd.DataFrame(rows)
            log.info(f"  Dept {dept} : {len(df_dept)} communes")
            frames.append(df_dept)

    if not frames:
        log.error("Aucune commune collectée — arrêt impossible si vide")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    log.info(f"  Total : {len(df)} communes")
    return df


def collect_sirene(
    departements: List[str],
    api_key:      str,
    timeout:      int = 15,
    max_results:  int = 2000,
) -> pd.DataFrame:
    """
    Collecte les établissements via l'API SIRENE INSEE.
    Source : https://portail-api.insee.fr/
    Données : siret, état (actif/fermé), APE, tranche effectifs, date création
    Clé     : gratuite sur portail-api.insee.fr

    Sans clé : retourne DataFrame vide → fallback simulé dans merge_all_sources()
    """
    if not api_key:
        log.warning("[COLLECTE] SIRENE : clé absente → données simulées")
        return pd.DataFrame()

    log.info(f"[COLLECTE] SIRENE — {len(departements)} département(s)")
    headers = {"X-INSEE-Api-Key-Integration": api_key}
    frames  = []

    for dept in departements:
        all_etabs, cursor = [], "*"

        while True:
            data = get_with_retry(
                "https://api.insee.fr/api-sirene/3.11/siret",
                headers=headers,
                params={
                    "q":       f"codeDepartementEtablissement:{dept}",
                    "nombre":  min(max_results, 1000),
                    "curseur": cursor,
                    "champs": (
                        "siret,siren,etatAdministratifEtablissement,"
                        "dateCreationEtablissement,"
                        "activitePrincipaleEtablissement,"
                        "adresseEtablissement,"
                        "trancheEffectifsEtablissement"
                    ),
                },
                timeout=timeout,
            )
            if not data:
                break

            etabs = data.get("etablissements", [])
            if not etabs:
                break

            for e in etabs:
                adr = e.get("adresseEtablissement", {})
                all_etabs.append({
                    "code_commune":     str(adr.get("codeCommuneEtablissement", "")).strip(),
                    "siret":            e.get("siret"),
                    "siren":            e.get("siren"),
                    "etat":             e.get("etatAdministratifEtablissement", "A"),
                    "date_creation":    e.get("dateCreationEtablissement"),
                    "code_ape":         e.get("activitePrincipaleEtablissement", {}).get("code", ""),
                    "tranche_effectif": e.get("trancheEffectifsEtablissement"),
                })

            next_cursor = data.get("header", {}).get("curseurSuivant")
            if not next_cursor or next_cursor == cursor or len(etabs) < 1000:
                break
            cursor = next_cursor

        if all_etabs:
            df_dept = pd.DataFrame(all_etabs)
            log.info(f"  Dept {dept} : {len(df_dept)} établissements")
            frames.append(df_dept)

    if not frames:
        log.warning("SIRENE : aucune donnée récupérée → fallback simulé")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    log.info(f"  Total SIRENE : {len(df)} établissements")
    return df


def collect_transport(departements: List[str] = None, timeout: int = 15) -> pd.DataFrame:
    """
    Collecte les données transport en commun — MULTI-SOURCES avec fallback.
    Stratégie (ordre de priorité) :
      1. API IDF Mobilités (open data officiel)
      2. API data.gouv.fr — référentiel gares SNCF
      3. API Overpass (OpenStreetMap)
      4. Fallback intégré — base statique 91+94
    """
    log.info("[COLLECTE] Transport — tentative multi-sources")
    departements = departements or []

    # ── SOURCE 1 : IDF Mobilités ─────────────────────────────────────────────
    df_idf = get_csv_with_retry(
        "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets"
        "/emplacement-des-gares-idf/exports/csv?lang=fr&timezone=Europe%2FParis",
        sep=";", timeout=timeout,
    )
    if not df_idf.empty:
        col_ville = next((c for c in ["nom_commune", "commune", "ville", "libelle_commune"]
                          if c in df_idf.columns), None)
        if col_ville:
            agg = df_idf.groupby(col_ville).size().reset_index(name="nb_gares")
            agg = agg.rename(columns={col_ville: "ville"})
            agg["lignes"] = "IDF Mobilités"
            log.info(f"  ✅ IDF Mobilités : {len(agg)} communes")
            return agg

    # ── SOURCE 2 : data.gouv.fr — gares SNCF ────────────────────────────────
    df_sncf = get_csv_with_retry(
        "https://static.data.gouv.fr/resources/liste-des-gares/20230302-122757/referentiel-gares-voyageurs.csv",
        sep=";", timeout=timeout,
    )
    if not df_sncf.empty and departements:
        col_dept    = next((c for c in ["departement_code", "code_dept", "departement"] if c in df_sncf.columns), None)
        col_commune = next((c for c in ["commune", "ville", "commune_libellee"] if c in df_sncf.columns), None)
        col_ligne   = next((c for c in ["ligne", "libelle_ligne"] if c in df_sncf.columns), None)
        if col_dept and col_commune:
            df_sncf[col_dept] = df_sncf[col_dept].astype(str).str.zfill(2)
            df_filtered = df_sncf[df_sncf[col_dept].isin(departements)]
            agg = df_filtered.groupby(col_commune).size().reset_index(name="nb_gares")
            agg = agg.rename(columns={col_commune: "ville"})
            if col_ligne:
                lignes_agg = (
                    df_filtered.groupby(col_commune)[col_ligne]
                    .apply(lambda x: ", ".join(sorted(set(x.dropna().astype(str)))))
                    .reset_index()
                    .rename(columns={col_commune: "ville", col_ligne: "lignes"})
                )
                agg = agg.merge(lignes_agg, on="ville", how="left")
            else:
                agg["lignes"] = "SNCF"
            agg["ville"] = agg["ville"].astype(str).str.strip()
            log.info(f"  ✅ Gares SNCF : {len(agg)} communes")
            return agg

    # ── SOURCE 3 : Overpass (OpenStreetMap) ──────────────────────────────────
    if departements:
        overpass_query = "[out:json][timeout:30];(" + "".join(
            f'area["ref:INSEE"~"^{d}"]["admin_level"="6"]->.a{d};'
            f'node["railway"~"station|halt"](area.a{d});'
            for d in departements
        ) + ");out body;"
        try:
            r = requests.post("https://overpass-api.de/api/interpreter",
                              data={"data": overpass_query}, timeout=35)
            if r.status_code == 200:
                elements = r.json().get("elements", [])
                if elements:
                    rows = []
                    for e in elements:
                        tags  = e.get("tags", {})
                        ville = tags.get("addr:city") or tags.get("is_in:city") or tags.get("name", "")
                        ligne = tags.get("network", "") or tags.get("operator", "")
                        if ville:
                            rows.append({"ville": ville.strip(), "ligne": ligne})
                    if rows:
                        df_osm = pd.DataFrame(rows)
                        agg = df_osm.groupby("ville").agg(
                            nb_gares=("ligne", "count"),
                            lignes=("ligne", lambda x: ", ".join(sorted(set(x.dropna()))))
                        ).reset_index()
                        log.info(f"  ✅ OpenStreetMap : {len(agg)} communes")
                        return agg
        except Exception as e:
            log.warning(f"  Overpass indisponible : {e}")

    # ── SOURCE 4 : Fallback statique intégré 91+94 ───────────────────────────
    log.info("  ✅ Fallback statique 91+94")
    GARES_STATIQUES = [
        # ESSONNE (91) — RER B
        ("Massy","Massy-Palaiseau","RER B/C, TGV"),("Palaiseau","Palaiseau","RER B"),
        ("Orsay","Orsay-Ville","RER B"),("Gif-sur-Yvette","Gif-sur-Yvette","RER B"),
        ("Bures-sur-Yvette","Bures-sur-Yvette","RER B"),("Igny","Igny","RER B"),
        # RER C
        ("Juvisy-sur-Orge","Juvisy","RER C/D"),("Athis-Mons","Athis-Mons","RER C"),
        ("Savigny-sur-Orge","Savigny-sur-Orge","RER C"),("Brétigny-sur-Orge","Brétigny-sur-Orge","RER C"),
        ("Arpajon","Arpajon","RER C"),("Étampes","Étampes","RER C, Ter"),
        ("Viry-Châtillon","Viry-Châtillon","RER C"),("Grigny","Grigny-Centre","RER C"),
        ("Longjumeau","Longjumeau","RER C"),("Sainte-Geneviève-des-Bois","Sainte-Geneviève-des-Bois","RER C"),
        # RER D
        ("Ris-Orangis","Ris-Orangis","RER D"),("Évry-Courcouronnes","Évry-Courcouronnes","RER D"),
        ("Corbeil-Essonnes","Corbeil-Essonnes","RER D"),("Mennecy","Mennecy","RER D"),
        ("Montgeron","Montgeron - Crosne","RER D"),("Brunoy","Brunoy","RER D"),
        ("Yerres","Yerres","RER D"),("Vigneux-sur-Seine","Vigneux-sur-Seine","RER D"),
        # VAL-DE-MARNE (94) — RER A
        ("Vincennes","Vincennes","RER A"),("Fontenay-sous-Bois","Fontenay-sous-Bois","RER A"),
        ("Nogent-sur-Marne","Nogent-sur-Marne","RER A"),("Joinville-le-Pont","Joinville-le-Pont","RER A"),
        ("Champigny-sur-Marne","Champigny-sur-Marne","RER A"),("Bry-sur-Marne","Bry-sur-Marne","RER A"),
        ("Boissy-Saint-Léger","Boissy-Saint-Léger","RER A"),("Saint-Maur-des-Fossés","Saint-Maur - Créteil","RER A"),
        ("Sucy-en-Brie","Sucy - Bonneuil","RER A"),
        # RER C
        ("Choisy-le-Roi","Choisy-le-Roi","RER C"),("Villeneuve-le-Roi","Villeneuve-le-Roi","RER C"),
        ("Vitry-sur-Seine","Vitry-sur-Seine","RER C"),("Orly","Orly-ville","RER C"),
        # RER D
        ("Villeneuve-Saint-Georges","Villeneuve-Saint-Georges","RER D"),
        ("Alfortville","Alfortville","RER D"),("Maisons-Alfort","Maisons-Alfort Alfortville","RER D"),
        # Métro
        ("Créteil","Créteil-Préfecture","Métro 8"),("Villejuif","Villejuif-Louis Aragon","Métro 7"),
        ("Ivry-sur-Seine","Mairie d'Ivry","Métro 7"),("Saint-Mandé","Saint-Mandé","Métro 1"),
    ]
    df_static = pd.DataFrame(GARES_STATIQUES, columns=["ville", "gare", "ligne"])
    agg = df_static.groupby("ville").agg(
        nb_gares=("gare", "nunique"),
        lignes=("ligne", lambda x: ", ".join(sorted(set(", ".join(x).split(", ")))))
    ).reset_index()
    log.info(f"  ✅ Fallback statique : {len(agg)} communes avec transport")
    return agg


def collect_dvf_sample(departements: List[str], timeout: int = 15) -> pd.DataFrame:
    """
    Collecte DVF (Demandes de Valeurs Foncières) — marché immobilier.
    Source : https://files.data.gouv.fr/geo-dvf/
    Optimisation : usecols pour limiter la mémoire sur des fichiers volumineux.
    """
    log.info("[COLLECTE] DVF — marché immobilier")
    current_year = datetime.now().year
    frames = []

    COLS_UTILES = ["code_commune", "valeur_fonciere", "surface_reelle_bati", "type_local"]

    for year in [current_year - 1, current_year - 2]:
        for dept in departements:
            url = (
                f"https://files.data.gouv.fr/geo-dvf/latest/csv/"
                f"{year}/departements/{dept}.csv.gz"
            )
            try:
                df_raw = pd.read_csv(
                    url,
                    compression="gzip",
                    low_memory=False,
                    usecols=lambda c: c in COLS_UTILES,
                )
                df_raw.columns = [c.lower().strip() for c in df_raw.columns]
                df_raw["annee"] = year
                log.info(f"  DVF {dept} {year} : {len(df_raw):,} transactions")
                frames.append(df_raw)
                time.sleep(0.3)  # Politesse API
            except Exception as e:
                log.warning(f"  DVF {dept} {year} : {e}")

    if not frames:
        log.warning("  DVF : aucune donnée disponible")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    df_immo = df[
        df.get("type_local", pd.Series()).isin(["Appartement", "Maison"])
    ].copy()

    df_immo["valeur_fonciere"]     = pd.to_numeric(df_immo["valeur_fonciere"],     errors="coerce")
    df_immo["surface_reelle_bati"] = pd.to_numeric(df_immo["surface_reelle_bati"], errors="coerce")
    df_immo = df_immo[
        (df_immo["valeur_fonciere"]    > 1000) &
        (df_immo["surface_reelle_bati"] > 5)   &
        (df_immo["surface_reelle_bati"] < 500)
    ]
    df_immo["prix_m2"] = df_immo["valeur_fonciere"] / df_immo["surface_reelle_bati"]

    agg = (
        df_immo.groupby("code_commune")
        .agg(prix_m2_median=("prix_m2", "median"), nb_transactions=("prix_m2", "count"))
        .reset_index()
    )
    agg["code_commune"] = agg["code_commune"].astype(str).str.zfill(5)
    log.info(f"  DVF agrégé : {len(agg)} communes")
    return agg

## 🔧 PARTIE 4 : Données simulées (fallback réaliste)

In [ ]:
def _classify_commune_type(df: pd.DataFrame) -> pd.Series:
    """Classifie les communes en 5 types selon leur densité."""
    densite = df["population"] / df["surface_km2"].clip(lower=0.1)
    return pd.cut(
        densite,
        bins=[0, 50, 200, 1000, 5000, float("inf")],
        labels=["rural_isole", "rural", "periurbain", "urbain", "tres_urbain"],
    ).astype(str)


def generate_bpe_fallback(df_communes: pd.DataFrame) -> pd.DataFrame:
    """
    Génère des données BPE médicales et commerciales simulées.
    Calibrées sur les ratios nationaux DREES 2022 et BPE 2022 :
      - Médecins généralistes : ~109 pour 100 000 habitants (nationale)
      - Pharmacies : ~34 pour 100 000 habitants
      - Hôpitaux : uniquement dans les communes > 20 000 habitants

    ⚠️ DONNÉES SIMULÉES — Remplacer par votre fichier BPE réel.
    """
    log.info("  Génération données BPE simulées (réalistes, seed=42)…")
    np.random.seed(42)
    df = df_communes.copy()
    pop = df["population"].clip(lower=100).astype(float)
    type_com = _classify_commune_type(df)

    # Ratios DREES 2022 par type de commune
    RATIOS_MED_GEN = {"rural_isole": 70, "rural": 90, "periurbain": 100,
                      "urbain": 115, "tres_urbain": 130}
    RATIOS_PHARMA  = {"rural_isole": 25, "rural": 30, "periurbain": 33,
                      "urbain": 38, "tres_urbain": 40}

    base_gen    = type_com.map(RATIOS_MED_GEN).astype(float)
    base_pharma = type_com.map(RATIOS_PHARMA).astype(float)

    result = pd.DataFrame({"code_commune": df["code_commune"].values})

    # Médical
    result["nb_medecin_generaliste"]  = (pop * base_gen   / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_medecin_specialiste"]  = (pop * 50         / 100_000 * np.random.lognormal(0, 0.6, len(df))).clip(0).astype(int)
    result["nb_infirmier"]            = (pop * 150        / 100_000 * np.random.lognormal(0, 0.4, len(df))).clip(0).astype(int)
    result["nb_dentiste"]             = (pop * 65         / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_kinesitherapeute"]     = (pop * 80         / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_pharmacie"]            = (pop * base_pharma / 100_000 * np.random.lognormal(0, 0.4, len(df))).clip(0).astype(int)
    result["nb_hopital"]              = np.where(pop > 20_000, np.random.randint(0, 3, len(df)), 0)
    result["nb_urgences"]             = np.where(pop > 50_000, np.random.randint(0, 2, len(df)), 0)
    result["nb_sage_femme"]           = (pop * 20         / 100_000 * np.random.lognormal(0, 0.6, len(df))).clip(0).astype(int)
    result["nb_orthophoniste"]        = (pop * 30         / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)

    # Commercial
    result["nb_supermarche"]   = (pop * 8    / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_epicerie"]      = (pop * 15   / 100_000 * np.random.lognormal(0, 0.6, len(df))).clip(0).astype(int)
    result["nb_boulangerie"]   = (pop * 20   / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_boucherie"]     = (pop * 10   / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_banque"]        = (pop * 12   / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_poste"]         = (pop * 6    / 100_000 * np.random.lognormal(0, 0.4, len(df))).clip(0).astype(int).clip(lower=1)
    result["nb_ecole_primaire"]= (pop * 30   / 100_000 * np.random.lognormal(0, 0.4, len(df))).clip(0).astype(int)
    result["nb_college"]       = (pop * 8    / 100_000 * np.random.lognormal(0, 0.5, len(df))).clip(0).astype(int)
    result["nb_lycee"]         = (pop * 5    / 100_000 * np.random.lognormal(0, 0.6, len(df))).clip(0).astype(int)

    result["donnees_bpe_simulees"] = True
    log.info(f"  BPE simulé : {len(result)} communes")
    return result


def generate_insee_fallback(df_communes: pd.DataFrame) -> pd.DataFrame:
    """
    Génère des données INSEE socio-économiques réalistes.
    Calibrées sur INSEE 2021 (chômage 7.5%, revenu médian 2150€, pauvreté 14.5%).
    ⚠️ DONNÉES SIMULÉES — Remplacer par Filosofi INSEE dès disponible.
    """
    log.info("  Génération données INSEE simulées (réalistes, seed=42)…")
    np.random.seed(42)
    df = df_communes.copy()
    type_commune = _classify_commune_type(df)
    df["type_commune"] = type_commune

    BASE = {
        "rural_isole": {"chom": 10.5, "rev": 1_650, "pauv": 18.0},
        "rural":       {"chom":  9.0, "rev": 1_750, "pauv": 15.0},
        "periurbain":  {"chom":  8.0, "rev": 2_100, "pauv": 11.0},
        "urbain":      {"chom": 11.0, "rev": 1_950, "pauv": 16.0},
        "tres_urbain": {"chom": 13.0, "rev": 2_300, "pauv": 19.0},
    }

    df["taux_chomage"]  = type_commune.map({t: v["chom"] for t, v in BASE.items()}).astype(float)
    df["revenu_median"] = type_commune.map({t: v["rev"]  for t, v in BASE.items()}).astype(float)
    df["taux_pauvrete"] = type_commune.map({t: v["pauv"] for t, v in BASE.items()}).astype(float)

    df["taux_chomage"]  += np.random.normal(0, 2.5, len(df))
    df["revenu_median"] += np.random.normal(0, 250,   len(df))
    df["taux_pauvrete"] += np.random.normal(0, 3.5, len(df))

    df["taux_chomage"]  = df["taux_chomage"].clip(4.0, 28.0)
    df["revenu_median"] = df["revenu_median"].clip(1_100, 5_000)
    df["taux_pauvrete"] = df["taux_pauvrete"].clip(4.0, 45.0)
    df["donnees_insee_simulees"] = True

    return df


def generate_sirene_fallback(df_communes: pd.DataFrame) -> pd.DataFrame:
    """Génère des données SIRENE simulées (~30 entreprises actives / 1000 hab)."""
    log.info("  Génération données SIRENE simulées (seed=123)…")
    np.random.seed(123)
    df = df_communes.copy()
    pop = df["population"].clip(lower=100).astype(float)
    nb_actifs = (pop / 1_000 * 30 * np.random.lognormal(0, 0.4, len(df))).astype(int).clip(lower=1)
    taux_ferm = np.random.uniform(0.10, 0.40, len(df))
    nb_fermes = (nb_actifs * taux_ferm).astype(int).clip(lower=0)
    nb_total  = nb_actifs + nb_fermes
    return pd.DataFrame({
        "code_commune":            df["code_commune"].values,
        "nb_entreprises_actives":  nb_actifs,
        "nb_entreprises_fermees":  nb_fermes,
        "nb_entreprises_total":    nb_total,
        "taux_survie_entreprises": (nb_actifs / nb_total.clip(lower=1)).round(4),
    })


def generate_transport_fallback(df_communes: pd.DataFrame) -> pd.DataFrame:
    """Génère des données transport simulées (probabilité de gare ∝ log(pop))."""
    log.info("  Génération données transport simulées (seed=77)…")
    np.random.seed(77)
    df = df_communes.copy()
    pop = df["population"].clip(lower=100).astype(float)
    proba_gare = np.clip(np.log1p(pop) / 15, 0, 0.9)
    has_gare   = np.random.random(len(df)) < proba_gare
    nb_gares   = np.where(has_gare, np.random.randint(1, 5, len(df)), 0)
    return pd.DataFrame({"ville": df["ville"].values, "nb_gares": nb_gares.astype(int)})


def generate_dvf_fallback(df_communes: pd.DataFrame) -> pd.DataFrame:
    """Génère des données DVF simulées (prix proportionnel à la densité)."""
    log.info("  Génération données DVF simulées (seed=99)…")
    np.random.seed(99)
    df = df_communes.copy()
    densite  = (df["population"] / df["surface_km2"].clip(lower=0.1)).clip(lower=1)
    prix_base = 1_500 + 1_500 * (np.log1p(densite) / np.log1p(densite.max()))
    prix_m2   = (prix_base * np.random.lognormal(0, 0.25, len(df))).clip(800, 12_000)
    return pd.DataFrame({
        "code_commune":   df["code_commune"].values,
        "prix_m2_median": prix_m2.round(0).astype(int),
        "nb_transactions": (df["population"] / 200 * np.random.uniform(0.5, 2.0, len(df))).astype(int).clip(1),
    })

## 🏗️ PARTIE 5 : Agrégateurs internes & Fusion toutes sources

In [ ]:
def _aggregate_sirene(df_sirene: pd.DataFrame) -> pd.DataFrame:
    """Agrège le détail SIRENE (établissements) en indicateurs par commune."""
    df = df_sirene.copy()
    df["code_commune"] = df["code_commune"].astype(str).str.strip()
    df = df[df["code_commune"].str.len() == 5]

    actifs = df[df["etat"] == "A"].groupby("code_commune")["siret"].count().rename("nb_entreprises_actives")
    fermes = df[df["etat"] == "F"].groupby("code_commune")["siret"].count().rename("nb_entreprises_fermees")
    total  = df.groupby("code_commune")["siret"].count().rename("nb_entreprises_total")

    agg = pd.concat([actifs, fermes, total], axis=1).fillna(0).reset_index()
    agg["taux_survie_entreprises"] = np.where(
        agg["nb_entreprises_total"] > 0,
        (agg["nb_entreprises_actives"] / agg["nb_entreprises_total"]).round(4),
        np.nan,
    )
    return agg


def _aggregate_transport(df_transport: pd.DataFrame) -> pd.DataFrame:
    """Agrège les gares par ville et déduplique les arrêts partagés."""
    if df_transport.empty:
        return pd.DataFrame()

    col_ville = next(
        (c for c in ["nom_commune", "commune", "ville", "libelle_commune"] if c in df_transport.columns), None
    )
    if col_ville is None:
        log.warning(f"  Transport : colonne ville introuvable parmi {df_transport.columns.tolist()}")
        return pd.DataFrame()

    col_gare = next((c for c in ["gare", "arret", "nom_gare", "stop_name", "nom"] if c in df_transport.columns), None)
    if col_gare:
        agg = (
            df_transport.groupby(col_ville)
            .agg(nb_gares=(col_gare, "nunique"))
            .reset_index()
            .rename(columns={col_ville: "ville"})
        )
    else:
        agg = df_transport.groupby(col_ville).size().reset_index(name="nb_gares").rename(columns={col_ville: "ville"})

    agg["ville"] = agg["ville"].astype(str).str.strip()
    return agg


def _detect_outliers(df: pd.DataFrame) -> None:
    """Signale les valeurs aberrantes via méthode Tukey (IQR × 3)."""
    for col in df.select_dtypes(include=[np.number]).columns:
        s = df[col].dropna()
        if len(s) < 4:
            continue
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR = Q3 - Q1
        if IQR == 0:
            continue
        outliers = df[(df[col] < Q1 - 3 * IQR) | (df[col] > Q3 + 3 * IQR)]["ville"].tolist()
        if outliers:
            log.warning(f"  Outliers [{col}] : {outliers[:5]}{'…' if len(outliers) > 5 else ''}")


def merge_all_sources(
    df_communes:  pd.DataFrame,
    df_bpe:       pd.DataFrame,
    df_sirene:    pd.DataFrame,
    df_transport: pd.DataFrame,
    df_dvf:       pd.DataFrame,
    use_fallback: bool = True,
) -> pd.DataFrame:
    """
    Fusionne toutes les sources sur code_commune (clé principale INSEE).

    ORDRE DE FUSION (logique data engineering) :
      1. Communes      → socle territorial (filtrée, dédupliquée)
      2. BPE médecins  → source principale v4 (enrichit avant les API)
      3. SIRENE        → tissu économique
      4. Transport     → accessibilité
      5. DVF           → marché immobilier
      6. INSEE fallback → socio-économique (chômage, revenus, pauvreté)

    Chaque source manquante déclenche son fallback simulé si use_fallback=True.
    """
    log.info("[FUSION] Assemblage de toutes les sources…")
    df = df_communes.copy()

    # ── Filtres qualité socle communes ────────────────────────────────────────
    n_init = len(df)
    df = df[(df["population"] > 0) & (df["surface_km2"] > 0)]
    df = df.dropna(subset=["code_commune"]).drop_duplicates(subset=["code_commune"])
    log.info(f"  {len(df)}/{n_init} communes valides après filtres qualité")

    # ── 1. BPE médecins (source principale) ───────────────────────────────────
    if not df_bpe.empty:
        df = df.merge(df_bpe, on="code_commune", how="left")
        n_enrichies = df["nb_medecin_generaliste"].notna().sum()
        log.info(f"  BPE réel : {n_enrichies} communes enrichies en données médicales")
        # Fallback uniquement pour les communes sans correspondance BPE
        if n_enrichies < len(df) and use_fallback:
            communes_sans_bpe = df[df["nb_medecin_generaliste"].isna()].copy()
            df_bpe_sim = generate_bpe_fallback(communes_sans_bpe)
            for col in df_bpe_sim.columns:
                if col != "code_commune" and col in df.columns:
                    df[col] = df[col].fillna(
                        df.index.map(df_bpe_sim.set_index("code_commune").get(col, pd.Series()))
                    )
            log.info(f"  BPE : {len(communes_sans_bpe)} communes complétées par fallback simulé")
    elif use_fallback:
        df_bpe_sim = generate_bpe_fallback(df)
        df = df.merge(df_bpe_sim, on="code_commune", how="left")
        log.info("  BPE simulé fusionné (aucun fichier réel disponible)")
    else:
        for col in ["nb_medecin_generaliste", "nb_pharmacie", "nb_hopital", "nb_urgences"]:
            df[col] = 0
        df["donnees_bpe_simulees"] = False

    # ── 2. SIRENE ────────────────────────────────────────────────────────────
    if not df_sirene.empty:
        df_s = _aggregate_sirene(df_sirene)
        df   = df.merge(df_s, on="code_commune", how="left")
        log.info(f"  SIRENE réel : {df['nb_entreprises_actives'].notna().sum()} communes enrichies")
    elif use_fallback:
        df_s = generate_sirene_fallback(df)
        df   = df.merge(df_s, on="code_commune", how="left")
        log.info("  SIRENE simulé fusionné")
    else:
        for col in ["nb_entreprises_actives", "nb_entreprises_fermees",
                    "nb_entreprises_total", "taux_survie_entreprises"]:
            df[col] = np.nan

    # ── 3. Transport ──────────────────────────────────────────────────────────
    if not df_transport.empty:
        df_t = _aggregate_transport(df_transport)
        if not df_t.empty:
            df = df.merge(df_t, on="ville", how="left")
            log.info("  Transport réel fusionné")
        else:
            df["nb_gares"] = 0
    elif use_fallback:
        df_t = generate_transport_fallback(df)
        df   = df.merge(df_t, on="ville", how="left")
        log.info("  Transport simulé fusionné")
    else:
        df["nb_gares"] = 0

    # ── 4. DVF ────────────────────────────────────────────────────────────────
    if not df_dvf.empty:
        df = df.merge(df_dvf, on="code_commune", how="left")
        log.info(f"  DVF réel : {df['prix_m2_median'].notna().sum()} communes avec données immo")
    elif use_fallback:
        df_d = generate_dvf_fallback(df)
        df   = df.merge(df_d, on="code_commune", how="left")
        log.info("  DVF simulé fusionné")
    else:
        df["prix_m2_median"]  = np.nan
        df["nb_transactions"] = np.nan

    # ── 5. INSEE socio-économique ─────────────────────────────────────────────
    if use_fallback:
        df_insee = generate_insee_fallback(df)
        for col in ["taux_chomage", "revenu_median", "taux_pauvrete",
                    "type_commune", "donnees_insee_simulees"]:
            if col in df_insee.columns:
                df[col] = df_insee[col].values
    else:
        for col in ["taux_chomage", "revenu_median", "taux_pauvrete"]:
            df[col] = np.nan
        df["type_commune"]           = "inconnu"
        df["donnees_insee_simulees"] = False

    # ── Nettoyage final des colonnes numériques ───────────────────────────────
    count_cols = ["nb_entreprises_actives", "nb_entreprises_fermees",
                  "nb_entreprises_total", "nb_gares",
                  "nb_medecin_generaliste", "nb_pharmacie", "nb_hopital"]
    for col in count_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0).astype(int)
        else:
            df[col] = 0

    df.columns = [c.lower().strip().replace(" ", "_").replace("-", "_") for c in df.columns]
    _detect_outliers(df)
    log.info(f"  Résultat fusion : {len(df)} communes × {len(df.columns)} colonnes")
    return df

## 🎯 PARTIE 6 : Feature engineering

In [ ]:
def normalize_robust(series: pd.Series, clip_pct: float = 99.0) -> pd.Series:
    """
    Normalisation min-max robuste avec écrêtage au percentile 99/1.
    Conserve les NaN pour les gérer dans weighted_score.
    """
    s = series.copy().astype(float)
    upper = np.nanpercentile(s, clip_pct)
    lower = np.nanpercentile(s, 100 - clip_pct)
    s = s.clip(lower=lower, upper=upper)
    s_min, s_max = float(np.nanmin(s)), float(np.nanmax(s))
    if s_max == s_min:
        return pd.Series(np.where(s.notna(), 0.5, np.nan), index=series.index)
    return (s - s_min) / (s_max - s_min)


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering territorial — V4 avec indicateurs médicaux.

    FEATURES PRODUITES :
    ── Démographie ──────────────────────────────────────────────────────
    densite_hab_km2       : Densité réelle (hab/km²)
    log_population        : log(pop+1) pour atténuer l'effet taille

    ── Tissu économique ─────────────────────────────────────────────────
    entreprises_1000hab   : Densité entreprises → vitalité économique locale
    dynamique_eco         : (actifs - fermés) / total × 100 → momentum éco

    ── Transport ────────────────────────────────────────────────────────
    gares_10k_hab         : Gares pour 10 000 habitants

    ── Immobilier ───────────────────────────────────────────────────────
    tension_immo          : prix_m2 normalisé → attractivité territoire

    ── Médical (nouveautés v4) ───────────────────────────────────────────
    medecins_10k_hab      : Médecins généralistes pour 10 000 habitants
    est_desert_medical    : 1 si medecins_10k_hab < seuil OMS (~2.5/10k)
    score_acces_soins     : composite médecins + pharmacies + hôpitaux

    ── Commercial (nouveautés v4) ────────────────────────────────────────
    score_acces_commercial: composite supermarchés + boulangeries + services
    est_desert_commercial : 1 si score_acces_commercial < seuil

    ── Normalisation [0,1] ───────────────────────────────────────────────
    norm_* : features normalisées pour le scoring
    """
    log.info("[FEATURES] Construction des variables…")
    df  = df.copy()
    pop = df["population"].clip(lower=1).astype(float)

    # ── Démographie
    df["densite_hab_km2"] = pop / df["surface_km2"].clip(lower=0.1).astype(float)
    df["log_population"]  = np.log1p(pop)

    # ── Tissu économique
    actifs = df["nb_entreprises_actives"].astype(float)
    fermes = df["nb_entreprises_fermees"].astype(float)
    total  = df["nb_entreprises_total"].astype(float).clip(lower=1)
    df["entreprises_1000hab"]       = (actifs * 1_000) / pop
    df["dynamique_eco"]             = ((actifs - fermes) * 100 / total).clip(-100, 100)
    if "taux_survie_entreprises" not in df.columns or df["taux_survie_entreprises"].isna().all():
        df["taux_survie_entreprises"] = (actifs / total).clip(0, 1)

    # ── Transport
    df["gares_10k_hab"] = (df["nb_gares"].astype(float) * 10_000 / pop).clip(0, 20)

    # ── Immobilier
    df["tension_immo"] = df["prix_m2_median"].astype(float) if "prix_m2_median" in df.columns else np.nan

    # ── Médical (source principale v4)
    nb_med_gen  = df["nb_medecin_generaliste"].astype(float) if "nb_medecin_generaliste" in df.columns else 0.0
    nb_pharma   = df["nb_pharmacie"].astype(float)           if "nb_pharmacie"           in df.columns else 0.0
    nb_hopital  = df["nb_hopital"].astype(float)             if "nb_hopital"             in df.columns else 0.0
    nb_infirmier= df["nb_infirmier"].astype(float)           if "nb_infirmier"           in df.columns else 0.0

    df["medecins_10k_hab"]   = (nb_med_gen * 10_000 / pop).clip(0, 100)
    df["pharmacies_10k_hab"] = (nb_pharma  * 10_000 / pop).clip(0, 50)

    # Seuil OMS adapté France : désert médical < 2.5 généralistes/10k hab
    SEUIL_DESERT_MEDICAL = 2.5
    df["est_desert_medical"] = (df["medecins_10k_hab"] < SEUIL_DESERT_MEDICAL).astype(int)

    # Score accès soins composite [0,1] — pondéré par importance
    df["score_acces_soins"] = (
        0.50 * normalize_robust(df["medecins_10k_hab"])     +
        0.25 * normalize_robust(df["pharmacies_10k_hab"])   +
        0.15 * normalize_robust(nb_hopital)                  +
        0.10 * normalize_robust(nb_infirmier)
    ).clip(0, 1)

    # ── Commercial
    nb_supermarche  = df["nb_supermarche"].astype(float)  if "nb_supermarche"  in df.columns else 0.0
    nb_boulangerie  = df["nb_boulangerie"].astype(float)  if "nb_boulangerie"  in df.columns else 0.0
    nb_banque       = df["nb_banque"].astype(float)       if "nb_banque"       in df.columns else 0.0
    nb_poste        = df["nb_poste"].astype(float)        if "nb_poste"        in df.columns else 0.0

    # Densité commerciale / 1000 hab
    df["supermarches_10k_hab"] = (nb_supermarche * 10_000 / pop).clip(0, 50)
    df["score_acces_commercial"] = (
        0.35 * normalize_robust(nb_supermarche + df.get("nb_epicerie", pd.Series(0, index=df.index)).astype(float)) +
        0.25 * normalize_robust(nb_boulangerie)  +
        0.25 * normalize_robust(nb_banque)       +
        0.15 * normalize_robust(nb_poste)
    ).clip(0, 1)

    SEUIL_DESERT_COMMERCIAL = 0.20
    df["est_desert_commercial"] = (df["score_acces_commercial"] < SEUIL_DESERT_COMMERCIAL).astype(int)

    # ── Normalisation pour scoring
    norm_map = {
        "densite_hab_km2":         "norm_densite",
        "entreprises_1000hab":     "norm_entreprises",
        "dynamique_eco":           "norm_dynamique_eco",
        "taux_survie_entreprises": "norm_survie",
        "gares_10k_hab":           "norm_transport",
        "log_population":          "norm_population",
        "tension_immo":            "norm_tension_immo",
        "score_acces_soins":       "norm_soins",
        "score_acces_commercial":  "norm_commercial",
    }
    for src, dst in norm_map.items():
        if src in df.columns and df[src].notna().any():
            df[dst] = normalize_robust(df[src])
        else:
            df[dst] = pd.Series(0.5, index=df.index)

    # Socio-économique
    df["norm_chomage"]       = normalize_robust(df["taux_chomage"])  if "taux_chomage"  in df.columns and df["taux_chomage"].notna().any()  else pd.Series(0.5, index=df.index)
    df["norm_taux_pauvrete"] = normalize_robust(df["taux_pauvrete"]) if "taux_pauvrete" in df.columns and df["taux_pauvrete"].notna().any() else pd.Series(0.5, index=df.index)
    df["norm_revenu_inv"]    = (1 - normalize_robust(df["revenu_median"])) if "revenu_median" in df.columns and df["revenu_median"].notna().any() else pd.Series(0.5, index=df.index)

    log.info(f"  {len(df.columns)} colonnes produites")
    return df

## 🗺️ PARTIE 7 : Scoring bancaire (8 scores)

In [ ]:
def weighted_score(df: pd.DataFrame, weights: Dict[str, float]) -> pd.Series:
    """
    Score pondéré [0,1]. Colonnes absentes → valeur neutre 0.5.
    La somme des poids est normalisée automatiquement.
    """
    result       = np.zeros(len(df))
    total_weight = 0.0
    for col, weight in weights.items():
        values       = df[col].fillna(0.5).clip(0, 1).values if col in df.columns else np.full(len(df), 0.5)
        result       += values * weight
        total_weight += weight
    return pd.Series((result / max(total_weight, 1e-9)).clip(0, 1), index=df.index)


def compute_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcule les 8 scores bancaires territoriaux.

    ┌──────────────────────────┬──────────────────────────────────────────────┐
    │ Score                    │ Usage bancaire                               │
    ├──────────────────────────┼──────────────────────────────────────────────┤
    │ score_fragilite          │ Médiation, microcrédit, accompagnement       │
    │ score_emergence          │ Prospection PME, développement agence        │
    │ score_freins_invisibles  │ Produits inclusifs, offres adaptées          │
    │ score_momentum           │ Décision investissement 3-5 ans              │
    │ potentiel_investissement │ Campagnes immo, assurance, épargne           │
    │ risque_credit_local      │ Provisions collectives, calibrage taux       │
    │ score_desert_medical     │ Orientation politique sociale agence         │
    │ score_desert_commercial  │ Analyse risque activité locale               │
    └──────────────────────────┴──────────────────────────────────────────────┘
    """
    log.info("[SCORING] Calcul des 8 scores bancaires…")
    df = df.copy()

    # 1. FRAGILITÉ — intègre désormais l'accès aux soins (v4)
    df["score_fragilite"] = weighted_score(df, {
        "norm_chomage":       0.30,
        "norm_taux_pauvrete": 0.25,
        "norm_revenu_inv":    0.20,
        "norm_dynamique_eco": 0.10,
        "norm_soins":         0.15,  # Désert médical = facteur de fragilité documenté
    })
    # Inverser norm_soins pour fragilité (peu de soins = plus fragile)
    df["score_fragilite"] = (
        0.30 * df["norm_chomage"].fillna(0.5) +
        0.25 * df["norm_taux_pauvrete"].fillna(0.5) +
        0.20 * df["norm_revenu_inv"].fillna(0.5) +
        0.10 * df["norm_dynamique_eco"].fillna(0.5) +
        0.15 * (1 - df["norm_soins"].fillna(0.5))  # Inversion : moins de soins = plus fragile
    ).clip(0, 1)

    df["fragilite_fiable"] = (
        "taux_chomage"  in df.columns and df["taux_chomage"].notna().any() and
        "taux_pauvrete" in df.columns and df["taux_pauvrete"].notna().any()
    )

    # 2. ÉMERGENCE
    df["score_emergence"] = weighted_score(df, {
        "norm_entreprises":   0.35,
        "norm_dynamique_eco": 0.25,
        "norm_transport":     0.20,
        "norm_tension_immo":  0.10,
        "norm_densite":       0.10,
    })

    # 3. FREINS INVISIBLES — désert médical intégré comme frein structurel
    df["norm_transport_inv"]   = 1 - df["norm_transport"].fillna(0.5)
    df["norm_entreprises_inv"] = 1 - df["norm_entreprises"].fillna(0.5)
    df["norm_soins_inv"]       = 1 - df["norm_soins"].fillna(0.5)
    df["score_freins_invisibles"] = (
        0.25 * df["norm_transport_inv"]   +
        0.20 * df["norm_entreprises_inv"] +
        0.25 * df["norm_taux_pauvrete"].fillna(0.5) +
        0.15 * df["norm_chomage"].fillna(0.5) +
        0.15 * df["norm_soins_inv"]       # Désert médical = frein invisible majeur
    ).clip(0, 1)

    # 4. MOMENTUM
    df["score_momentum"] = (
        0.50 * df["score_emergence"].fillna(0.5) +
        0.30 * (1 - df["score_fragilite"].fillna(0.5)) +
        0.20 * df["norm_dynamique_eco"].fillna(0.5)
    ).clip(0, 1)

    # 5. POTENTIEL INVESTISSEMENT
    df["potentiel_investissement"] = (
        0.40 * df["score_emergence"].fillna(0.5) +
        0.30 * df["score_momentum"].fillna(0.5) +
        0.20 * df["norm_densite"].fillna(0.5) +
        0.10 * (1 - df["score_freins_invisibles"].fillna(0.5))
    ).clip(0, 1)

    # 6. RISQUE CRÉDIT LOCAL
    df["risque_credit_local"] = (
        0.40 * df["score_fragilite"].fillna(0.5) +
        0.30 * df["score_freins_invisibles"].fillna(0.5) +
        0.20 * (1 - df["norm_survie"].fillna(0.5)) +
        0.10 * (1 - df["score_momentum"].fillna(0.5))
    ).clip(0, 1)

    df["classe_risque"] = np.select(
        [df["risque_credit_local"] < 0.30,
         df["risque_credit_local"] < 0.50,
         df["risque_credit_local"] < 0.70],
        ["FAIBLE", "MODERE", "ELEVE"],
        default="CRITIQUE",
    )

    # 7. DÉSERT MÉDICAL (score continu basé sur BPE)
    df["score_desert_medical"] = (1 - df["norm_soins"].fillna(0.5)).clip(0, 1)
    df["niveau_desert_medical"] = np.select(
        [df["score_desert_medical"] < 0.25,
         df["score_desert_medical"] < 0.50,
         df["score_desert_medical"] < 0.75],
        ["bien_desservi", "correct", "sous-doté"],
        default="desert_medical",
    )

    # 8. DÉSERT COMMERCIAL (score continu basé sur BPE)
    df["score_desert_commercial"] = (1 - df["norm_commercial"].fillna(0.5)).clip(0, 1)
    df["niveau_desert_commercial"] = np.select(
        [df["score_desert_commercial"] < 0.25,
         df["score_desert_commercial"] < 0.50,
         df["score_desert_commercial"] < 0.75],
        ["bien_desservi", "correct", "sous-doté"],
        default="desert_commercial",
    )

    # Log des stats par score
    for score in ["score_fragilite", "score_emergence", "score_freins_invisibles",
                  "score_momentum", "potentiel_investissement", "risque_credit_local",
                  "score_desert_medical", "score_desert_commercial"]:
        s = df[score].dropna()
        if len(s):
            log.info(f"  {score:<35} : med={s.median():.3f}  min={s.min():.3f}  max={s.max():.3f}")

    return df

## 📊 PARTIE 8 : Segmentation territoriale

In [ ]:
def apply_segmentation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Classification territoriale en 5 segments (règles hiérarchiques explicites).

    Hiérarchie des règles (ordre prioritaire) :
      1. territoire_bloque    → freins très élevés ET forte fragilité
      2. territoire_attractif → fort potentiel ET fort momentum
      3. territoire_emergent  → forte émergence ET bon momentum
      4. territoire_fragile   → fragilité significative
      5. territoire_stable    → cas général

    Actions bancaires par segment :
      bloque    → Microcrédit, médiation bancaire, pas de crédit standard
      fragile   → Accompagnement renforcé, produits adaptés, vigilance
      stable    → Maintien offre standard, fidélisation
      emergent  → Prospection PME, crédit investissement, développement
      attractif → Offres premium, immobilier, épargne, assurance vie
    """
    log.info("[SEGMENTATION] Classification territoriale…")
    df = df.copy()

    def classify(row) -> str:
        def v(key: str) -> float:
            val = row.get(key, 0.5)
            return float(val) if (val is not None and val == val) else 0.5

        fri = v("score_freins_invisibles")
        fra = v("score_fragilite")
        em  = v("score_emergence")
        mom = v("score_momentum")
        pot = v("potentiel_investissement")

        if fri > 0.68 and fra > 0.55:
            return "territoire_bloque"
        if pot > 0.65 and mom > 0.60:
            return "territoire_attractif"
        if em  > 0.55 and mom > 0.45:
            return "territoire_emergent"
        if fra > 0.50:
            return "territoire_fragile"
        return "territoire_stable"

    df["segment"] = df.apply(classify, axis=1)

    for seg, count in df["segment"].value_counts().items():
        log.info(f"  {seg:<25} : {count:4} communes ({100*count/len(df):.1f}%)")

    return df

## 📤 PARTIE 9 : Export

In [ ]:
EXPORT_COLS = [
    # Identifiants
    "code_commune", "ville", "code_dept", "code_postal",
    "latitude", "longitude", "population", "surface_km2",
    "densite_hab_km2", "type_commune",
    # Scores principaux
    "score_fragilite", "score_emergence", "score_freins_invisibles",
    "score_momentum", "potentiel_investissement", "risque_credit_local",
    "classe_risque", "segment", "fragilite_fiable",
    # Scores déserts (nouveauté v4 — issus BPE)
    "score_desert_medical", "niveau_desert_medical",
    "score_desert_commercial", "niveau_desert_commercial",
    # Économie
    "nb_entreprises_actives", "nb_entreprises_total",
    "taux_survie_entreprises", "entreprises_1000hab", "dynamique_eco",
    # Transport
    "nb_gares", "gares_10k_hab",
    # Immobilier
    "prix_m2_median", "nb_transactions",
    # Social
    "taux_chomage", "revenu_median", "taux_pauvrete",
    # Médical (BPE — source principale v4)
    "nb_medecin_generaliste", "nb_medecin_specialiste",
    "nb_pharmacie", "nb_hopital", "nb_urgences",
    "nb_infirmier", "nb_dentiste", "nb_kinesitherapeute",
    "medecins_10k_hab", "pharmacies_10k_hab",
    "est_desert_medical", "score_acces_soins",
    # Commercial (BPE)
    "nb_supermarche", "nb_epicerie", "nb_boulangerie",
    "nb_banque", "nb_poste", "nb_boucherie",
    "nb_ecole_primaire", "nb_college", "nb_lycee",
    "est_desert_commercial", "score_acces_commercial",
    # Qualité données
    "donnees_insee_simulees", "donnees_bpe_simulees",
]


def _to_python_type(val):
    """Convertit les types numpy en types Python natifs pour la sérialisation JSON."""
    if val is None:
        return None
    try:
        if pd.isna(val):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(val, np.integer):
        return int(val)
    if isinstance(val, np.floating):
        return float(val)
    if isinstance(val, np.bool_):
        return bool(val)
    return val


def generate_report(df: pd.DataFrame) -> str:
    """Rapport textuel complet : statistiques, top10, corrélations, déserts v4."""
    sep = "=" * 72
    lines = [sep, "RAPPORT D'ANALYSE TERRITORIALE — PIPELINE ICEBERG V4", sep, ""]

    lines += [
        "STATISTIQUES GÉNÉRALES",
        f"  Communes analysées  : {len(df)}",
        f"  Départements        : {', '.join(sorted(df['code_dept'].astype(str).unique()))}",
        f"  Population totale   : {int(df['population'].sum()):,} habitants",
        "",
    ]

    lines.append("DISTRIBUTION DES SEGMENTS")
    for seg, cnt in df["segment"].value_counts().items():
        lines.append(f"  {seg:<25} : {cnt:4} ({100*cnt/len(df):.1f}%)")
    lines.append("")

    lines.append("DISTRIBUTION DES CLASSES DE RISQUE")
    for r, cnt in df["classe_risque"].value_counts().items():
        lines.append(f"  {r:<15} : {cnt:4} ({100*cnt/len(df):.1f}%)")
    lines.append("")

    # Déserts médicaux (nouveauté v4)
    if "niveau_desert_medical" in df.columns:
        lines.append("DÉSERTS MÉDICAUX (données BPE)")
        for niv, cnt in df["niveau_desert_medical"].value_counts().items():
            lines.append(f"  {niv:<20} : {cnt:4} ({100*cnt/len(df):.1f}%)")
        if "medecins_10k_hab" in df.columns:
            lines.append(f"  Médecins/10k moy : {df['medecins_10k_hab'].mean():.1f}")
        lines.append("")

    lines.append("TOP 10 — POTENTIEL D'INVESTISSEMENT")
    for i, (_, row) in enumerate(df.nlargest(10, "potentiel_investissement").iterrows(), 1):
        lines.append(
            f"  {i:2}. {row['ville']:<30} ({row.get('code_dept','')}) "
            f"score={row['potentiel_investissement']:.3f}  [{row.get('segment','')}]"
        )
    lines.append("")

    lines.append("TOP 10 — DÉSERTS MÉDICAUX")
    if "score_desert_medical" in df.columns:
        for i, (_, row) in enumerate(df.nlargest(10, "score_desert_medical").iterrows(), 1):
            lines.append(
                f"  {i:2}. {row['ville']:<30} ({row.get('code_dept','')}) "
                f"score_desert={row['score_desert_medical']:.3f}  "
                f"med/10k={row.get('medecins_10k_hab', 0):.1f}"
            )
    lines.append("")

    lines.append("TOP 10 — RISQUE CRÉDIT LOCAL LE PLUS ÉLEVÉ")
    for i, (_, row) in enumerate(df.nlargest(10, "risque_credit_local").iterrows(), 1):
        lines.append(
            f"  {i:2}. {row['ville']:<30} ({row.get('code_dept','')}) "
            f"risque={row['risque_credit_local']:.3f}  [{row.get('classe_risque','')}]"
        )
    lines.append("")

    lines += ["", sep]
    return "\n".join(lines)


def export_data(
    df:          pd.DataFrame,
    json_path:   str,
    csv_path:    str,
    excel_path:  str = None,
    report_path: str = None,
) -> None:
    """Export CSV + Excel + JSON (avec métadonnées v4) + rapport texte."""
    cols      = [c for c in EXPORT_COLS if c in df.columns]
    df_export = df[cols].copy()
    float_cols = df_export.select_dtypes(include=[float]).columns
    df_export[float_cols] = df_export[float_cols].round(4)

    df_export.to_csv(csv_path, index=False, encoding="utf-8-sig")
    log.info(f"  CSV   : {csv_path}")

    if excel_path:
        try:
            df_export.to_excel(excel_path, index=False, engine="openpyxl")
            log.info(f"  Excel : {excel_path}")
        except Exception as e:
            log.warning(f"  Excel non exporté : {e}")

    metadata = {
        "version":      "4.0.0",
        "pipeline":     "iceberg-territorial-scoring",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "nb_communes":  len(df_export),
        "departements": df["code_dept"].unique().tolist() if "code_dept" in df.columns else [],
        "scores":       ["score_fragilite", "score_emergence", "score_freins_invisibles",
                         "score_momentum", "potentiel_investissement", "risque_credit_local",
                         "score_desert_medical", "score_desert_commercial"],
        "nouveautes_v4": ["BPE médecins intégré EN AMONT", "score_desert_medical",
                          "score_desert_commercial", "score_acces_soins",
                          "medecins_10k_hab", "pharmacies_10k_hab"],
        "segments":     df["segment"].value_counts().to_dict() if "segment" in df.columns else {},
        "risques":      df["classe_risque"].value_counts().to_dict() if "classe_risque" in df.columns else {},
        "donnees_simulees": bool(df.get("donnees_bpe_simulees", pd.Series([False])).any()),
        "avertissement_legal": (
            "risque_credit_local est un indicateur de contexte territorial uniquement. "
            "Ne peut pas être utilisé seul pour une décision d'octroi. RGPD applicable."
        ),
    }

    records = [
        {col: _to_python_type(val) for col, val in row.items()}
        for _, row in df_export.iterrows()
    ]

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"metadata": metadata, "data": records}, f, ensure_ascii=False, indent=2)
    log.info(f"  JSON  : {json_path} ({Path(json_path).stat().st_size / 1024:.1f} Ko)")

    if report_path:
        report = generate_report(df)
        with open(report_path, "w", encoding="utf-8") as f:
            f.write(report)
        log.info(f"  Rapport : {report_path}")
        print("\n" + report)

## 🚀 PARTIE 10 : Orchestrateur principal

```
ORDRE D'EXÉCUTION DU PIPELINE :

  [ÉTAPE 1/7]  load_and_prepare_bpe()      ← SOURCE PRINCIPALE (local → API → fallback)
  [ÉTAPE 2/7]  collect_communes()          ← Geo API (socle territorial)
  [ÉTAPE 3/7]  collect_sirene()            ← API INSEE (tissu économique)
               collect_transport()         ← IDF Mobilités / SNCF / OSM
               collect_dvf_sample()        ← data.gouv.fr (immo)
  [ÉTAPE 4/7]  merge_all_sources()         ← Fusion BPE en premier
  [ÉTAPE 5/7]  build_features()            ← Feature engineering
  [ÉTAPE 6/7]  compute_scores()            ← 8 scores bancaires
               apply_segmentation()        ← 5 segments
  [ÉTAPE 7/7]  export_data()               ← CSV / JSON / Excel / Rapport
```

In [ ]:
def run_pipeline(config: dict = None) -> Optional[pd.DataFrame]:
    """
    Orchestre le pipeline V4 de A à Z en 7 étapes.

    Différence clé avec V3 :
      → Le fichier BPE (médecins) est chargé EN PREMIER (étape 1),
        avant tout appel API, comme source principale structurante.
      → Aucun appel réseau inutile si le fichier local est fourni.
      → 8 scores au lieu de 6, incluant score_desert_medical et score_desert_commercial.

    Retourne le DataFrame final ou None en cas d'échec critique.
    """
    if config is None:
        config = CONFIG

    config.setdefault("sirene_max_results", 2000)
    config.setdefault("use_fallback_data",  True)
    config.setdefault("api_timeout",        15)
    config.setdefault("output_excel",       None)
    config.setdefault("output_report",      "iceberg_rapport_v4.txt")
    config.setdefault("bpe_fichier_local",  "")
    config.setdefault("bpe_separateur",     ";")

    start = time.time()
    SEP = "=" * 70
    log.info(SEP)
    log.info("DÉMARRAGE — PIPELINE ICEBERG V4")
    log.info(f"  Départements        : {config['departements']}")
    log.info(f"  Fichier BPE local   : {config['bpe_fichier_local'] or '(aucun → API/fallback)'}")
    log.info(f"  Fallback data       : {config['use_fallback_data']}")
    log.info(SEP)

    # ── ÉTAPE 1 : BPE médecins — SOURCE PRINCIPALE (avant tout appel API) ────
    log.info("\n[ÉTAPE 1/7] Chargement BPE — données médicales & équipements")
    df_bpe = load_and_prepare_bpe(
        departements       = config["departements"],
        bpe_fichier_local  = config["bpe_fichier_local"],
        bpe_separateur     = config["bpe_separateur"],
        timeout            = config["api_timeout"],
    )

    # ── ÉTAPE 2 : Communes (socle territorial obligatoire) ────────────────────
    log.info("\n[ÉTAPE 2/7] Collecte des communes (Geo API)")
    df_communes = collect_communes(config["departements"], config["api_timeout"])
    if df_communes.empty:
        log.error("Communes vides — arrêt du pipeline")
        return None

    # ── ÉTAPE 3 : Sources API secondaires (parallélisables si besoin) ─────────
    log.info("\n[ÉTAPE 3/7] Collecte des sources API secondaires")
    df_sirene = collect_sirene(
        config["departements"],
        config["sirene_api_key"],
        config["api_timeout"],
        config["sirene_max_results"],
    )
    df_transport = collect_transport(config["departements"], config["api_timeout"])
    df_dvf       = collect_dvf_sample(config["departements"], config["api_timeout"])

    # ── ÉTAPE 4 : Fusion toutes sources (BPE intégrée en premier) ─────────────
    log.info("\n[ÉTAPE 4/7] Fusion & nettoyage (BPE en ancre principale)")
    df = merge_all_sources(
        df_communes  = df_communes,
        df_bpe       = df_bpe,
        df_sirene    = df_sirene,
        df_transport = df_transport,
        df_dvf       = df_dvf,
        use_fallback = config["use_fallback_data"],
    )

    # ── ÉTAPE 5 : Feature engineering ─────────────────────────────────────────
    log.info("\n[ÉTAPE 5/7] Feature engineering")
    df = build_features(df)

    # ── ÉTAPE 6 : Scoring & segmentation ──────────────────────────────────────
    log.info("\n[ÉTAPE 6/7] Scoring (8 scores) & segmentation")
    df = compute_scores(df)
    df = apply_segmentation(df)

    # ── ÉTAPE 7 : Export ──────────────────────────────────────────────────────
    log.info("\n[ÉTAPE 7/7] Export des données")
    export_data(
        df,
        json_path   = config["output_json"],
        csv_path    = config["output_csv"],
        excel_path  = config.get("output_excel"),
        report_path = config.get("output_report"),
    )

    # ── Résumé final ──────────────────────────────────────────────────────────
    elapsed = time.time() - start
    log.info(f"\nPIPELINE V4 TERMINÉ en {elapsed:.1f}s")
    log.info(SEP)
    log.info(f"  Communes analysées : {len(df)}")
    log.info(f"  JSON               : {config['output_json']}")
    log.info(f"  CSV                : {config['output_csv']}")
    if config.get("output_excel"):
        log.info(f"  Excel              : {config['output_excel']}")
    bpe_source = "fichier local" if config["bpe_fichier_local"] and df_bpe is not None and not df_bpe.empty else "API/simulé"
    log.info(f"  Source BPE         : {bpe_source}")
    log.info(SEP)

    return df

## ▶️ Lancer le pipeline

> **Avec votre fichier BPE local :**
> Renseignez `bpe_fichier_local` avec le chemin vers votre CSV médecins.
> Le pipeline le chargera **avant tout appel API**.
>
> **Sans fichier local :**
> Le pipeline tentera l'API INSEE BPE, puis simulera les données si indisponible.

In [ ]:
config = {
    "departements":       ["91", "94"],

    # ─── SOURCE PRINCIPALE : fichier CSV médecins ─────────────────────────
    # Lu EN PREMIER, avant tout appel API.
    # Si vide → téléchargement automatique depuis data.iledefrance.fr
    # Source officielle :
    # https://data.iledefrance.fr/explore/dataset/
    # les-service-de-sante-par-commune-ou-par-arrondissement-base-permanente-des-equip/
    "bpe_fichier_local":  "",   # Vide = téléchargement auto depuis data.iledefrance.fr
    "bpe_separateur":     None,   # None = détection automatique ( , ou ; )

    # ─── API & sorties ────────────────────────────────────────────────────
    "sirene_api_key":     "",    # Clé gratuite : https://portail-api.insee.fr/
    "output_json":        "iceberg_dataset_v4.json",
    "output_csv":         "iceberg_dataset_v4.csv",
    "output_excel":       "iceberg_dataset_v4.xlsx",
    "output_report":      "iceberg_rapport_v4.txt",
    "api_timeout":        15,
    "sirene_max_results": 2000,
    "use_fallback_data":  True,
}

df_result = run_pipeline(config)


## 📊 Aperçu des résultats

In [ ]:
if df_result is not None:
    display_cols = [
        "ville", "code_dept", "population", "type_commune",
        "score_fragilite", "score_emergence", "potentiel_investissement",
        "risque_credit_local", "classe_risque", "segment",
        # Colonnes BPE — source principale v4
        "nb_medecin_generaliste", "medecins_10k_hab",
        "score_desert_medical", "niveau_desert_medical",
    ]
    display_cols = [c for c in display_cols if c in df_result.columns]
    display(df_result[display_cols].head(20))
    print(f"\n{len(df_result)} communes · {len(df_result.columns)} colonnes")

## 🏥 Analyse des déserts médicaux & commerciaux (données BPE)

In [ ]:
if df_result is not None:

    print("=" * 60)
    print("DÉSERTS MÉDICAUX (source BPE)")
    print("=" * 60)
    if "niveau_desert_medical" in df_result.columns:
        print(df_result["niveau_desert_medical"].value_counts().to_string())
    print(f"\nScore moyen : {df_result['score_desert_medical'].mean():.3f}")
    print(f"Médecins/10k hab moyen : {df_result['medecins_10k_hab'].mean():.1f}")

    print("\nTop 10 déserts médicaux :")
    top_med_cols = [c for c in ["ville", "code_dept", "score_desert_medical",
                                 "niveau_desert_medical", "nb_medecin_generaliste",
                                 "medecins_10k_hab", "nb_pharmacie"] if c in df_result.columns]
    display(df_result.nlargest(10, "score_desert_medical")[top_med_cols])

    print("\n" + "=" * 60)
    print("DÉSERTS COMMERCIAUX (source BPE)")
    print("=" * 60)
    if "niveau_desert_commercial" in df_result.columns:
        print(df_result["niveau_desert_commercial"].value_counts().to_string())

    if "est_desert_medical" in df_result.columns and "est_desert_commercial" in df_result.columns:
        double = df_result[
            (df_result["est_desert_medical"] == 1) &
            (df_result["est_desert_commercial"] == 1)
        ]
        print(f"\n⚠️  Communes en double désert (médical + commercial) : {len(double)}")
        if len(double) > 0:
            display(double[["ville", "code_dept", "population",
                            "score_desert_medical", "score_desert_commercial"]].head(10))

## 📈 Statistiques par segment

In [ ]:
if df_result is not None:
    agg_cols = {
        "nb_communes":          ("ville", "count"),
        "pop_moy":              ("population", "mean"),
        "fragilite_moy":        ("score_fragilite", "mean"),
        "emergence_moy":        ("score_emergence", "mean"),
        "potentiel_moy":        ("potentiel_investissement", "mean"),
        "risque_moy":           ("risque_credit_local", "mean"),
        "desert_medical_moy":   ("score_desert_medical", "mean"),
    }
    agg_cols_filtered = {k: v for k, v in agg_cols.items() if v[0] in df_result.columns}
    agg = df_result.groupby("segment").agg(**agg_cols_filtered).round(3)
    display(agg)
    print("\nDistribution risque :")
    print(df_result["classe_risque"].value_counts())